# Clase 4 — Redes neuronales y frameworks

## Pregunta central

> **¿Qué aprende una red neuronal y qué trabajo resuelve un framework?**

## Idea principal

Una red transforma tensores mediante capas con parámetros; entrenar consiste en ajustar esos parámetros para reducir una pérdida.

## Objetivos de aprendizaje

Al finalizar la clase deberías poder:

- Explicar neurona, capa, activación, parámetro y forward pass.
- Describir conceptualmente backpropagation, epoch y batch.
- Distinguir el rol general de MLP, CNN y Transformer.
- Reconocer tensores, formas y dispositivos.
- Entrenar el mismo clasificador pequeño con PyTorch y Keras.

## Recorrido de la clase

| Paso | Tema |
|---:|---|
| 1 | De una neurona a una red |
| 2 | Cómo aprende una red |
| 3 | MLP, CNN y Transformer |
| 4 | Tensores y dispositivos |
| 5 | Práctica equivalente en PyTorch y Keras |
| 6 | Actividad: cambiar capacidad |

## Cómo trabajar con este notebook

1. Ejecutá las celdas en el orden propuesto.
2. Antes de modificar código, observá y describí el resultado.
3. Cambiá solamente las variables marcadas con `TODO`.
4. No es necesario implementar algoritmos desde cero.
5. Si aparece un término nuevo, buscá primero su definición en el glosario de la clase.

**Conexión con el programa:** Deep Learning de ambos tracks; PyTorch será el framework operativo principal.

---
## 1. De una neurona a una red

Una red neuronal es una función compuesta por operaciones numéricas.
El nombre está inspirado en la biología, pero una neurona artificial
no reproduce una neurona humana ni posee intención.

Una unidad artificial realiza tres operaciones:

```text
entradas
  x1 ----(peso w1)---                                  +--> suma + bias --> activación --> salida
  x2 ----(peso w2)---/
```

Si `x` contiene las entradas y `w` los pesos, la combinación puede
resumirse como `x · w + b`. La **activación** permite representar
relaciones que una sola recta no podría describir.

| Parte | Qué hace | Qué debemos evitar asumir |
|---|---|---|
| Peso | Multiplica una entrada | No demuestra importancia causal |
| Bias | Desplaza la combinación | No es un “error” o sesgo social |
| Activación | Aplica una transformación | No toma una decisión humana |
| Capa | Ejecuta varias unidades en paralelo | No necesariamente representa un concepto nombrable |
| Parámetro | Valor aprendido: pesos y bias | No lo elegimos manualmente uno por uno |
| Hiperparámetro | Configuración: capas, neuronas, learning rate | No se aprende de la misma forma |

### Por qué necesitamos activaciones

Si apiláramos solamente transformaciones lineales, muchas capas
seguirían siendo equivalentes a una sola transformación lineal. Las
activaciones no lineales permiten construir fronteras más complejas.

| Activación | Idea básica | Uso introductorio |
|---|---|---|
| ReLU | Convierte valores negativos en cero | Capas ocultas |
| Sigmoid | Lleva un valor al rango 0–1 | Algunas salidas binarias |
| Softmax | Convierte varios logits en una distribución | Salida multiclase |

Apilar capas forma una red:

```text
64 valores de entrada
        |
        v
capa oculta de 32 unidades
        |
      ReLU
        |
        v
capa de salida de 10 unidades
        |
        v
     logits
```

Los **logits** son puntajes sin normalizar. Para clasificación,
`softmax` puede convertirlos en valores entre 0 y 1 que suman uno.
Elegir el mayor produce la clase predicha.

```text
logits:        [1.2, -0.3, 2.1]
softmax:       [0.27, 0.06, 0.67]
clase elegida: índice 2
```

## Glosario mínimo

| Término | Explicación breve |
|---|---|
| Tensor | Arreglo numérico con una o más dimensiones |
| Shape | Tamaño del tensor en cada dimensión |
| Forward pass | Cálculo que produce una predicción |
| Loss | Número que cuantifica el error de entrenamiento |
| Backpropagation | Cálculo de cómo contribuyó cada parámetro al error |
| Gradient descent | Actualización de parámetros para reducir la loss |
| Batch | Grupo de muestras procesado en una actualización |
| Epoch | Recorrido completo por los datos de entrenamiento |
| Learning rate | Tamaño de cada actualización |
| Inferencia | Forward pass sin actualizar parámetros |
| Optimizador | Regla que actualiza parámetros usando gradientes |
| Logit | Puntaje de salida antes de convertirlo en probabilidad |
| Arquitectura | Organización de capas y conexiones |
| Framework | Conjunto integrado de herramientas para definir, entrenar, guardar y ejecutar modelos |
| Autodiferenciación | Cálculo automático de gradientes a partir de las operaciones realizadas |

---
## 2. Cómo aprende una red, sin magia

Entrenar significa buscar parámetros que reduzcan un error sobre
ejemplos de train. Cada batch atraviesa este ciclo:

```text
batch de entradas + labels
          |
          v
1. forward pass -> logits y predicciones
          |
          v
2. loss -> diferencia numérica con los labels
          |
          v
3. backpropagation -> gradiente de cada parámetro
          |
          v
4. optimizador -> actualización de parámetros
          |
          +------------------------> siguiente batch
```

### Loss

La loss transforma el error del batch en un número. Para
clasificación multiclase usaremos cross-entropy, que penaliza al
modelo cuando asigna poca probabilidad a la clase correcta.

- loss alta: las predicciones están lejos del objetivo;
- loss descendente: el optimizador está mejorando sobre train;
- loss baja en train: no garantiza buen resultado en validation.

### Gradiente y backpropagation

Un gradiente indica en qué dirección local cambiaría la loss si
modificáramos un parámetro. Backpropagation recorre las operaciones
desde la loss hacia las capas anteriores y calcula esos gradientes.

```text
parámetro -> operaciones -> predicción -> loss
    ^                                    |
    |--------- gradiente hacia atrás ----|
```

Backpropagation no “entiende” la imagen. Aplica cálculo sobre un
grafo de operaciones. El optimizador decide cómo usar los gradientes.

### Batch, epoch y learning rate

| Concepto | Qué controla | Ejemplo |
|---|---|---|
| Batch | Muestras usadas en una actualización | 64 imágenes |
| Epoch | Un recorrido por todo train | 8 recorridos |
| Learning rate | Tamaño aproximado de cada paso | 0.01 |

Si train tiene 1 152 muestras y el batch es 64, una epoch contiene
aproximadamente 18 actualizaciones.

- learning rate muy alto: el entrenamiento puede ser inestable;
- demasiado bajo: puede avanzar muy lentamente;
- más epochs: no garantiza mejor generalización;
- una red más grande: tiene más capacidad, pero también puede
  memorizar ruido y cuesta más ejecutar.

### Entrenamiento frente a inferencia

| Operación | Entrenamiento | Inferencia |
|---|---:|---:|
| Forward pass | Sí | Sí |
| Labels necesarios | Sí | No |
| Loss | Sí | No |
| Gradientes | Sí | No |
| Actualización de parámetros | Sí | No |

Durante inferencia no se actualizan parámetros. Esto reduce trabajo
y evita que el modelo cambie por recibir una solicitud.

## 3. Un mapa de arquitecturas

Una arquitectura define cómo se organizan las capas y qué
conexiones puede usar el modelo.

| Arquitectura | Qué estructura aprovecha | Primera asociación |
|---|---|---|
| MLP | Cada entrada puede conectarse con cada unidad | Datos tabulares o vectores |
| CNN | Vecindad local y filtros compartidos | Imágenes y espectrogramas |
| Transformer | Relaciones entre posiciones mediante attention | Texto, visión y audio |

### MLP

Una Multi-Layer Perceptron recibe un vector. En la práctica de
dígitos aplanaremos una imagen de 8 × 8:

```text
matriz 8 x 8 -> vector de 64 valores -> MLP
```

Al aplanar perdemos la estructura explícita de filas y columnas.

### CNN

Una Convolutional Neural Network aplica filtros locales sobre la
grilla y comparte los mismos parámetros en distintas posiciones.
Esto aprovecha que píxeles cercanos suelen estar relacionados.

### Transformer

Un transformer representa cada elemento y usa attention para
combinar información de otras posiciones. Lo veremos con más
detalle en la clase 6.

No son compartimentos estancos. Un sistema puede usar una CNN para
extraer rasgos visuales y un Transformer para relacionarlos. En esta
clase usamos una **MLP** porque deja visible el mecanismo común sin
sumar todavía convoluciones ni attention.

---
## 4. Tensores y dispositivos

### Qué resuelve un framework

Un **framework de Deep Learning** es una biblioteca integrada que
evita implementar desde cero la infraestructura numérica de una red.
PyTorch y TensorFlow/Keras ofrecen piezas equivalentes con
interfaces distintas:

| Responsabilidad | Ejemplo |
|---|---|
| Tensores | Almacenar y operar arreglos numéricos |
| Autodiferenciación | Calcular gradientes para backpropagation |
| Capas y losses | Reutilizar componentes probados |
| Optimizadores | Actualizar parámetros |
| Dispositivos | Mover cálculos entre CPU y aceleradores |
| Carga de datos | Formar y mezclar batches |
| Serialización | Guardar y volver a cargar pesos |

El framework automatiza operaciones; el equipo todavía decide el
problema, los datos, la arquitectura, la evaluación y los controles.

### Tensores

Los frameworks organizan entradas, parámetros y salidas como
tensores. Un tensor es un arreglo numérico con:

- shape;
- dtype;
- dispositivo;
- valores.

La forma es parte del contrato:

```text
32 imágenes de 8 × 8 -> shape (32, 8, 8)
imágenes aplanadas    -> shape (32, 64)
salida para 10 clases -> shape (32, 10)
```

En `(32, 10)`, cada una de las 32 filas contiene 10 logits. Un error
de shape suele indicar que una capa esperaba otra organización.

### Dtype

- Las entradas de redes suelen representarse con decimales
  `float32`.
- Los labels de clase suelen ser enteros.
- Usar un dtype incorrecto puede causar un error o consumir memoria
  innecesaria.

### Dispositivo

PyTorch y TensorFlow pueden ejecutar en CPU o aceleradores. Los
datos y el modelo deben estar en dispositivos compatibles.

```text
modelo en CPU + tensor en CPU -> operación válida
modelo en GPU + tensor en CPU -> error de dispositivo
```

Para esta nivelación usamos CPU: el dataset es pequeño y el
resultado es reproducible en una notebook común.

### Dataset de dígitos

Cada muestra es una imagen de 8 × 8 en escala de grises. Los valores
originales van de 0 a 16 y se normalizan al rango 0–1. El label es
un entero entre 0 y 9.

Train ajusta la red, validation permite comparar capacidad y test se
reserva para la evaluación final.

En esta demostración, PyTorch y Keras ejecutan la misma arquitectura
sobre el mismo test para verificar que ambas interfaces producen el
contrato esperado. No usaremos esas accuracies para cambiar capas,
epochs ni hiperparámetros; la actividad de selección usa validation.
En un proyecto real, cualquier comparación que cambie una decisión
debe hacerse antes de abrir test.

In [ ]:
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

FAST_MODE = True
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

digits = load_digits()
X = (digits.images / 16.0).astype("float32")
y = digits.target.astype("int64")
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.20,
    stratify=y_train_val,
    random_state=SEED,
)

fig, axes = plt.subplots(1, 8, figsize=(11, 2))
for ax, image, label in zip(axes, X_train[:8], y_train[:8]):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.suptitle("Entradas: cada dígito es una matriz de 8 × 8")
plt.tight_layout()
plt.show()

print(
    "Train:", X_train.shape,
    "Validation:", X_val.shape,
    "Test:", X_test.shape,
)

---
## 5. Práctica A — PyTorch como framework principal

El ciclo está preparado para que podamos leer sus cinco pasos sin
tener que escribirlo desde cero. La red recibe 64 píxeles y produce
10 logits, uno por dígito.

### Componentes que aparecerán en el código

| Componente PyTorch | Responsabilidad |
|---|---|
| `TensorDataset` | Une entradas y labels |
| `DataLoader` | Crea batches y mezcla train |
| `nn.Module` | Define la arquitectura |
| `CrossEntropyLoss` | Calcula la loss multiclase |
| `Adam` | Actualiza parámetros |
| `model.train()` | Activa modo de entrenamiento |
| `model.eval()` | Activa modo de evaluación |
| `torch.no_grad()` | Evita calcular gradientes en inferencia |

`model.train()` no ejecuta entrenamiento por sí solo. Solo cambia el
comportamiento de capas que distinguen train de evaluación.

El ciclo explícito nos deja ver responsabilidades:

```text
cargar batch
    -> limpiar gradientes anteriores
    -> forward
    -> loss
    -> backward
    -> optimizer.step()
```

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))
device = torch.device("cpu")

X_train_t = torch.tensor(X_train.reshape(-1, 64))
y_train_t = torch.tensor(y_train)
X_val_t = torch.tensor(X_val.reshape(-1, 64))
X_test_t = torch.tensor(X_test.reshape(-1, 64))

loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

class RedDigitos(nn.Module):
    def __init__(self, hidden_units=32):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Linear(64, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, 10),
        )

    def forward(self, x):
        return self.capas(x)

def entrenar_pytorch(hidden_units=32, epochs=8):
    modelo = RedDigitos(hidden_units).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizador = torch.optim.Adam(modelo.parameters(), lr=0.01)
    historial = []

    for _ in range(epochs):
        modelo.train()
        perdida_acumulada = 0.0
        for xb, yb in loader:
            optimizador.zero_grad()       # 1. limpiar gradientes
            logits = modelo(xb.to(device)) # 2. forward
            loss = loss_fn(logits, yb.to(device))  # 3. error
            loss.backward()               # 4. backpropagation
            optimizador.step()             # 5. actualización
            perdida_acumulada += loss.item() * len(xb)
        historial.append(perdida_acumulada / len(loader.dataset))

    modelo.eval()
    with torch.no_grad():
        logits_val = modelo(X_val_t.to(device))
        pred_val = logits_val.argmax(dim=1).cpu().numpy()
        logits_test = modelo(X_test_t.to(device))
        pred_test = logits_test.argmax(dim=1).cpu().numpy()
    return (
        modelo,
        historial,
        logits_val.cpu(),
        pred_val,
        logits_test.cpu(),
        pred_test,
    )

inicio = time.perf_counter()
(
    modelo_torch,
    loss_torch,
    logits_val_torch,
    pred_val_torch,
    logits_torch,
    pred_torch,
) = entrenar_pytorch(hidden_units=32, epochs=8 if FAST_MODE else 20)
tiempo_torch = time.perf_counter() - inicio
parametros_torch = sum(p.numel() for p in modelo_torch.parameters())

pd.Series({
    "shape de salida": str(tuple(logits_torch.shape)),
    "parámetros": parametros_torch,
    "accuracy test": accuracy_score(y_test, pred_torch),
    "segundos": tiempo_torch,
}, name="PyTorch").to_frame()

### Qué conviene observar

- La salida tiene una fila por muestra y una columna por clase.
- `argmax` elige el índice del logit mayor.
- El número de parámetros depende de las conexiones entre capas.
- El modelo nunca recibe las imágenes de test durante entrenamiento.
- La pérdida debería bajar, pero el objetivo final sigue siendo
  generalizar a datos no usados para ajustar parámetros.

Para esta arquitectura, los parámetros provienen de:

```text
capa 1: 64 entradas x 32 unidades + 32 bias
capa 2: 32 entradas x 10 unidades + 10 bias
```

“Millones de parámetros” significa repetir esta idea a una escala
mucho mayor. No significa millones de reglas escritas a mano.

---
## 6. Práctica B — la misma intención con Keras

Keras ofrece una interfaz de más alto nivel: `compile`, `fit` y
`predict` encapsulan el ciclo. No buscamos decidir un ganador, sino
reconocer que cambian las interfaces y se mantienen los conceptos.

| Intención | PyTorch explícito | Keras |
|---|---|---|
| Definir capas | `nn.Sequential` | `keras.Sequential` |
| Elegir loss/optimizador | Objetos separados | `compile` |
| Recorrer batches | Bucle Python | `fit` |
| Obtener salidas | Llamar al modelo | `predict` |
| Historial de loss | Lista propia | Objeto `History` |

Una interfaz de alto nivel reduce código repetitivo. Una interfaz
explícita facilita personalizar ciclos complejos. La elección depende
del proyecto y del equipo, no de una superioridad universal.

In [ ]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import tensorflow as tf

tf.keras.utils.set_random_seed(SEED)
modelo_keras = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(64,)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(10),
])
modelo_keras.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=["accuracy"],
)

inicio = time.perf_counter()
historia_keras = modelo_keras.fit(
    X_train.reshape(-1, 64),
    y_train,
    epochs=8 if FAST_MODE else 20,
    batch_size=64,
    verbose=0,
)
logits_keras = modelo_keras.predict(
    X_test.reshape(-1, 64), verbose=0
)
tiempo_keras = time.perf_counter() - inicio
pred_keras = logits_keras.argmax(axis=1)

pd.Series({
    "shape de salida": str(logits_keras.shape),
    "parámetros": modelo_keras.count_params(),
    "accuracy test": accuracy_score(y_test, pred_keras),
    "segundos": tiempo_keras,
}, name="Keras").to_frame()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(loss_torch, marker="o", label="PyTorch")
plt.plot(historia_keras.history["loss"], marker="s", label="Keras")
plt.xlabel("Epoch")
plt.ylabel("Loss de entrenamiento")
plt.title("Dos interfaces, la misma señal que intentamos reducir")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

comparacion = pd.DataFrame([
    ["PyTorch", parametros_torch, accuracy_score(y_test, pred_torch)],
    ["Keras", modelo_keras.count_params(),
     accuracy_score(y_test, pred_keras)],
], columns=["framework", "parametros", "accuracy_test"])
comparacion.round(3)

### Preguntas de observación

1. ¿Por qué ambos modelos tienen la misma forma de salida?
2. ¿Los pesos aprendidos deberían ser idénticos? ¿Por qué no?
3. ¿La menor loss de train garantiza el mejor resultado en test?
4. ¿Qué parte del ciclo oculta `fit` y qué parte deja explícita
   PyTorch?

### Cómo leer la curva de loss

- Descenso sostenido: el optimizador está reduciendo error en train.
- Línea casi plana: el learning rate, la capacidad o los datos pueden
  limitar el aprendizaje.
- Oscilaciones grandes: el paso puede ser alto o los batches muy
  variables.

La gráfica muestra solamente loss de train. Para diagnosticar
overfitting necesitaríamos observar también validation.

---
## Actividad — cambiar la capacidad de la red

Modificá solo `HIDDEN_UNITS`. Compará parámetros, loss y accuracy.
Una red más grande no tiene garantizado un mejor resultado.

**Capacidad** es la flexibilidad del modelo para representar
relaciones. Aumentar unidades ocultas:

- aumenta parámetros;
- aumenta memoria y operaciones;
- puede capturar un patrón más complejo;
- puede facilitar overfitting;
- no reemplaza mejores datos.

Probá valores pequeños, intermedios y grandes manteniendo constantes
las demás decisiones. Así el experimento cambia una variable por vez.

In [ ]:
# TODO: probá 8, 32 o 128.
HIDDEN_UNITS = 8

(
    modelo_actividad,
    loss_actividad,
    _,
    pred_val_actividad,
    _,
    _,
) = entrenar_pytorch(
    hidden_units=HIDDEN_UNITS,
    epochs=6 if FAST_MODE else 15,
)
resultado_actividad = pd.Series({
    "hidden_units": HIDDEN_UNITS,
    "parámetros": sum(
        p.numel() for p in modelo_actividad.parameters()
    ),
    "loss final": loss_actividad[-1],
    "accuracy validation": accuracy_score(
        y_val, pred_val_actividad
    ),
})
resultado_actividad.round(3).to_frame("resultado")

Escribí una conclusión de dos frases: ¿la capacidad extra ayudó en
validation y qué costo agregó? Para comparar justamente, mantené
constantes los datos, la cantidad de epochs y la métrica. **No uses
test para elegir `HIDDEN_UNITS`**: test queda reservado para una sola
evaluación final después de decidir la arquitectura.

---

## Síntesis de la clase

- Una red es una composición de transformaciones con parámetros.
- Forward produce una salida; backpropagation calcula gradientes.
- Batch, epoch y learning rate describen el proceso de entrenamiento.
- MLP, CNN y Transformer incorporan estructuras útiles diferentes.
- PyTorch y Keras cambian la interfaz, no los conceptos fundamentales.

## Comprobación conceptual

Antes de continuar, intentá responder sin mirar el notebook:

1. ¿Cuál era el problema central de la clase?
2. ¿Qué entrada recibió el sistema y qué salida produjo?
3. ¿Qué decisión humana siguió siendo necesaria?
4. ¿Qué limitación observaste en el experimento?

Si podés explicarlo con tus propias palabras y justificarlo con un resultado visible, alcanzaste el objetivo introductorio.

## Puente con la próxima clase

La clase 5 aplica estas ideas a imágenes: convoluciones, modelos preentrenados, cajas y máscaras.

## Conexión con los tracks

Track Salud profundizará redes para audio, texto y predicción; Track Imagen entrenará y evaluará arquitecturas visuales con PyTorch.

La implementación profunda, el trabajo con datasets reales y las decisiones de producción se desarrollarán en los módulos especializados.